# Results analysis

Reproduces the operation results and the inference cost comparison reported in the paper.

1. Setup
2. Robot runs: loop latency (Table V), detections in operation (Table VI) and bounding box widths (Fig. 10)
3. Inference cost on the reference video (Table IV)

Section 2 runs on any machine. Section 3 loads the four models and processes video, so it needs a GPU runtime.

## 1. Setup

In [ ]:
import os
from pathlib import Path

# VS Code starts the notebook in analysis/; move to the repository root
if Path("../data/experiments").exists():
    os.chdir("..")
# On Colab, clone the repository
elif not Path("data/experiments").exists():
    !git clone -q https://github.com/vitor-domeniquelli/brazilian-traffic-sign-detection.git /content/repo
    %pip install -q ultralytics roboflow tensorflow
    os.chdir("/content/repo")

print(os.getcwd())

## 2. Robot runs

60 runs, 10 per sign. Each run folder holds `frames.csv` (one row per loop iteration), `meta.json` and `video.mp4`. The runs were recorded with an earlier version of `host/live_test.py` that used Portuguese field names, which are renamed on loading.

In [ ]:
import pandas as pd

ROOT = Path("data/experiments")

# Recorded field name -> name used in this notebook
COLUMNS = {
    "t_captura_ms": "t_capture_ms",
    "t_inferencia_ms": "t_inference_ms",
    "t_decisao_ms": "t_decision_ms",
    "t_comando_ms": "t_command_ms",
    "classe_detectada": "detected_class",
    "confianca": "confidence",
    "estado": "state",
    "comando_enviado": "command_sent",
}

frames = []
for csv_path in sorted(ROOT.glob("*/run*/frames.csv")):
    f = pd.read_csv(csv_path).rename(columns=COLUMNS)
    f["sign"] = csv_path.parent.parent.name
    f["run"] = csv_path.parent.name
    frames.append(f)
frames = pd.concat(frames, ignore_index=True)

print(f"runs: {frames.groupby(['sign', 'run']).ngroups}   loop iterations: {len(frames)}")

### 2.1 Loop latency (Table V)

`t_capture_ms` is the interval between consecutive loop iterations, that is, the loop period, and contains the other stages. `t_command_ms` is timed only in the iterations in which a command was sent.

In [ ]:
STAGES = {
    "t_capture_ms": "Loop period",
    "t_inference_ms": "Inference",
    "t_decision_ms": "Decision",
    "t_command_ms": "Command dispatch",
}

lat = frames[list(STAGES)].describe().T
lat["p95"] = frames[list(STAGES)].quantile(0.95)
lat.index = list(STAGES.values())
print(lat[["count", "mean", "std", "p95", "min", "max"]].round(2).to_string())

period = frames["t_capture_ms"].mean()
print(f"\nloop rate          : {1000 / period:.1f} iterations/s "
      f"(p95 period: {1000 / frames['t_capture_ms'].quantile(0.95):.1f} iterations/s)")
print(f"inference share    : {frames['t_inference_ms'].mean() / period:.0%}")
print(f"command share      : {frames['t_command_ms'].fillna(0).mean() / period:.0%} (averaged over all iterations)")

### 2.2 Detections in operation (Table VI)

A detection is correct when the detected class matches the sign of the run. Only detections with confidence of at least 0.70, the threshold used by the robot, were logged.

In [ ]:
det = frames[frames["detected_class"].notna()].copy()
det["correct"] = det["detected_class"] == det["sign"]

print(f"runs with a detection   : {det.groupby(['sign', 'run']).ngroups} of 60")
print(f"frames with a detection : {len(det)} of {len(frames)}")
print(f"correct class           : {det['correct'].sum()} ({det['correct'].mean():.1%})")
print(f"mean confidence         : correct {det.loc[det['correct'], 'confidence'].mean():.2f}, "
      f"wrong {det.loc[~det['correct'], 'confidence'].mean():.2f}")

In [ ]:
det["width_range"] = pd.cut(det["bbox_w"], [0, 40, 90, float("inf")],
                            labels=["up to 40", "41 to 90", "above 90"])
table = det.groupby("width_range", observed=False).agg(
    detections=("correct", "size"),
    wrong=("correct", lambda s: (~s).sum()))
table["error_rate_%"] = (100 * table["wrong"] / table["detections"]).round(1)
print(table.to_string())

print(f"\ndetections up to 40 px            : {(det['bbox_w'] <= 40).mean():.1%}")
print(f"smallest wrong detection above 90 : {det.loc[~det['correct'] & (det['bbox_w'] > 90), 'bbox_w'].min():.0f} px")

In [ ]:
# Raising the confidence threshold to the value that maximized F1 on the test set
THRESHOLD = 0.838
removed = det["confidence"] < THRESHOLD
n_wrong, n_correct = (~det["correct"]).sum(), det["correct"].sum()
print(f"wrong detections removed   : {(removed & ~det['correct']).sum()} of {n_wrong}")
print(f"correct detections removed : {(removed & det['correct']).sum()} of {n_correct} "
      f"({(removed & det['correct']).sum() / n_correct:.1%})")

### 2.3 Bounding box widths (Fig. 10)

Widths in pixels in the 640-pixel frame, from the dataset annotations and from the detections logged in the runs. The effective training widths resize each training box by 100 random factors between 0.5 and 1.5, the range of the scale augmentation used in training (`scale=0.5`).

In [ ]:
from getpass import getpass
from roboflow import Roboflow

# Any Roboflow account key works, since the dataset is public
rf = Roboflow(api_key="aVtYS205nG9XQsuNywb6")
ds = rf.workspace("vitor-chagas").project("robocar-dataset-c").version(2).download("yolov8")
DATASET = Path(ds.location)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# YOLO labels store the normalized width in the fourth field
widths = {}
for split in ["train", "valid", "test"]:
    w = np.array([float(line.split()[3]) * 640
                  for txt in (DATASET / split / "labels").glob("*.txt")
                  for line in txt.read_text().splitlines() if line.strip()])
    widths[split] = w
    print(f"{split:5s}: n={len(w)}  min={w.min():.0f}  p5={np.percentile(w, 5):.0f}  "
          f"median={np.median(w):.0f}  up to 40 px: {(w <= 40).mean():.1%}")

train = widths["train"]
rng = np.random.default_rng(0)
effective = (train[:, None] * rng.uniform(0.5, 1.5, (len(train), 100))).ravel()
operation = det["bbox_w"].to_numpy()
print(f"effective: median={np.median(effective):.0f}  up to 40 px: {(effective <= 40).mean():.1%}")

bins = np.arange(0, 340, 10)
pct = lambda x: np.full(len(x), 100 / len(x))   # each histogram sums to 100%
plt.figure(figsize=(6, 3.5))
plt.hist(effective, bins=bins, weights=pct(effective), color="lightblue",
         label="Training, after scale augmentation")
plt.hist(train, bins=bins, weights=pct(train), alpha=0.6,
         label=f"Training labels (n={len(train)})")
plt.hist(operation, bins=bins, weights=pct(operation), histtype="step", linewidth=1.5,
         label=f"Detections in operation (n={len(operation)})")
plt.xlabel("Bounding box width (px, 640-px frame)")
plt.ylabel("Boxes per 10-px bin (%)")
plt.legend()
plt.tight_layout()
plt.savefig("analysis/bbox_width_histogram.png", dpi=300)
plt.show()

## 3. Inference cost on the reference video (Table IV)

The four pipelines process the same frames, one frame at a time, after a warm-up inference. In the two-stage pipeline the Hough transform runs on the CPU and the classifier on the GPU. The timed interval covers detection and classification; drawing and writing the output video are outside it.

In [ ]:
%pip install -q tensorflow

In [ ]:
import time
import cv2
import tensorflow as tf
from ultralytics import YOLO

VIDEO = "data/reference_video.mp4"
MAX_FRAMES = 2000
SIZE = 80
results = []

for name, weights, out_path in [("YOLOv8n", "models/yolov8n.pt", "analysis/video_yolov8n.mp4"),
                                ("YOLO11n", "models/yolov11n.pt", "analysis/video_yolo11n.mp4")]:
    model = YOLO(weights)
    model.predict(np.zeros((720, 1280, 3), np.uint8), verbose=False)   # warm-up

    cap = cv2.VideoCapture(VIDEO)
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), cap.get(cv2.CAP_PROP_FPS),
                          (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
    times = []
    while len(times) < MAX_FRAMES:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        view = model.predict(frame, conf=0.5, verbose=False)[0].plot()
        times.append((time.perf_counter() - t0) * 1000)
        out.write(view)
    cap.release()
    out.release()

    t = np.array(times)
    params = sum(p.numel() for p in model.model.parameters())
    results.append({"model": name, "parameters_M": round(params / 1e6, 2), "mean_ms": round(t.mean(), 2),
                    "sd_ms": round(t.std(), 2), "p95_ms": round(np.percentile(t, 95), 2), "frames": len(t)})
    print(results[-1])

In [ ]:
for name, weights, out_path in [("Hough + MLP", "models/mlp.keras", "analysis/video_hough_mlp.mp4"),
                                ("Hough + CNN", "models/cnn.keras", "analysis/video_hough_cnn.mp4")]:
    model = tf.keras.models.load_model(weights)
    model(np.zeros((1, SIZE, SIZE, 1), np.float32), training=False)   # warm-up

    cap = cv2.VideoCapture(VIDEO)
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), cap.get(cv2.CAP_PROP_FPS),
                          (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
    times = []
    while len(times) < MAX_FRAMES:
        ok, frame = cap.read()
        if not ok:
            break
        t0 = time.perf_counter()
        view = frame.copy()
        gray = cv2.medianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), 3)
        c = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT_ALT, 1, 60,
                             param1=120, param2=0.9, minRadius=10, maxRadius=80)
        if c is not None:
            cx, cy, r = [int(v) for v in np.round(c[0][0])]
            half = int(r * 1.25)
            crop = gray[max(cy - half, 0):cy + half, max(cx - half, 0):cx + half]
            if crop.size:
                crop = cv2.resize(crop, (SIZE, SIZE))
                _, binary = cv2.threshold(crop, 0, 255, cv2.THRESH_OTSU)
                model((binary.astype(np.float32) / 255).reshape(1, SIZE, SIZE, 1), training=False)
                cv2.rectangle(view, (cx - half, cy - half), (cx + half, cy + half), (0, 255, 0), 2)
        times.append((time.perf_counter() - t0) * 1000)
        out.write(view)
    cap.release()
    out.release()

    t = np.array(times)
    results.append({"model": name, "parameters_M": round(model.count_params() / 1e6, 2), "mean_ms": round(t.mean(), 2),
                    "sd_ms": round(t.std(), 2), "p95_ms": round(np.percentile(t, 95), 2), "frames": len(t)})
    print(results[-1])

In [ ]:
print(pd.DataFrame(results).to_string(index=False))